In [ ]:
from os import listdir as ls
import pandas as pd
import seaborn as sns
from matplotlib.pyplot import subplots
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.inputs import get_gdps, get_country_pop
from emu_renewal.constants import OUTPUTS_PATH, ANALYSIS_NAMES
from emu_renewal.outputs import get_idatas_for_mob_type
from emu_renewal.utils import get_country_short_name
from emu_renewal.outputs import get_param_vals_by_analysis, get_ratios_from_disps, get_median_ratios
from emu_renewal.plotting import plot_exponent_dispersion_comparison
from emu_renewal.outputs import get_idatas_for_mob_type, get_median_ratios

set_matplotlib_formats("svg")

https://unstats.un.org/unsd/methodology/m49/overview/#

In [ ]:
job_path = OUTPUTS_PATH / "46693102"
all_countries = ls(job_path)
disp_posts = {c: get_param_vals_by_analysis("dispersion_proc", job_path / c) for c in all_countries}
ratio_dists = get_ratios_from_disps(disp_posts)

In [ ]:
import pycountry
import pandas as pd
import plotly.graph_objects as go

In [ ]:
def get_country_name(c):
    try:
        return pycountry.countries.lookup(c).name
    except:
        return c

In [ ]:
import numpy as np

In [ ]:
mob_source = "g_mob"
idatas, _ = get_idatas_for_mob_type(job_path, all_countries, mob_source)


In [ ]:
plot_df = pd.DataFrame(
    {
        "mobility exponent": {c: float(d.posterior["mob_exp"].median()) for c, d in idatas.items()},
        "dispersion ratio": get_median_ratios(ratio_dists, mob_source),
        "GDP per capita": get_gdps(2020),
        "population (millions)": {c: get_country_pop(c) / 1e6 for c in all_countries},
    }
)
plot_df["country name"] = plot_df.index.to_series().apply(get_country_name)
plot_df.loc[plot_df["population (millions)"].isna(), "population (millions)"] = 0.0

In [ ]:
fig = go.Figure(layout={"height": 600, "width": 750})
fig.add_trace(
    go.Scatter(
        x=plot_df["dispersion ratio"],
        y=plot_df["mobility exponent"],
        mode="markers",
        text=plot_df["country name"],
        hoverinfo="text",
        marker=dict(
            size=plot_df["population (millions)"], 
            line=dict(width=1.0, color="black"),
            color=plot_df["GDP per capita"],
            sizemode="area",
            sizemin=4.0,
            sizeref=2.5,
            colorscale="reds",
        ),
    ),
)